In [ ]:
from typing import Any
from pathlib import Path
import json

import polars as pl
from pydantic import BaseModel

IMG = "img"
NAME = "name"
KP_1 = "kp-1"

path_data = Path("data")
path_captures = path_data / "captures"
path_labels = path_data / "labels.csv"

# Solo agarro las columnas que necesito.
_COLS_TO_KEEP = [IMG, KP_1]
df = pl.read_csv(path_labels, columns=_COLS_TO_KEEP)

# Le saco el path agregado por label-studio.
df.insert_column(0, pl.Series(NAME, [img.split("-")[-1] for img in df[IMG]]))
df

In [ ]:
from typing import Dict, List, Literal

from pydantic import Field
import numpy as np

class Label(BaseModel):
    x: float
    y: float
    keypointlabels: Literal["foot", "reference"]
    original_width: float
    original_height: float


class LabelPoints(BaseModel):
    foot: List[Label] = Field(default_factory=list)
    reference: List[Label] = Field(default_factory=list)

    @property
    def foot_array(self) -> np.ndarray:
        return np.array([[f.x, f.y] for f in self.foot], dtype=np.float32)

    @property
    def reference_array(self) -> np.ndarray:
        return np.array([[r.x, r.y] for r in self.reference], dtype=np.float32)



In [ ]:
name2label: Dict[str, LabelPoints] = {}
for row in df.iter_rows(named=True):
    name = row[NAME]
    for label in json.loads(row[KP_1]):
        label["keypointlabels"] = label["keypointlabels"][0]
        label = Label(**label)
        if name not in name2label:
            name2label[name] = LabelPoints()
        
        if label.keypointlabels == "foot":
            name2label[name].foot.append(label)
        elif label.keypointlabels == "reference":
            name2label[name].reference.append(label)
        else:
            raise NotImplementedError("Implement.")

In [ ]:
IMG_NAME = "004.jpg"
display(name2label[IMG_NAME].foot)
display(name2label[IMG_NAME].reference)

In [ ]:
import cv2
import numpy as np

# Coordenadas en la imagen (x, y) de puntos de referencia.
#points_ref = name2label[IMG_NAME].reference_array
points_ref = np.array([     # FIXME: Hardcodeo por que al labelear no tuve en cuenta el orden de los puntos.
    [66, 15],
    [21, 45],
    [42, 41],
    [59, 60]
], dtype=np.float32)

# TODO: Ver si es esto.
#IMG_HEIGHT = name2label[IMG_NAME].reference[0].original_height
# Invertimos Y en puntos de imagen
#points_ref_flipped = points_ref.copy()
#points_ref_flipped[:, 1] = IMG_HEIGHT - points_ref_flipped[:, 1]
# TODO: Ver si es esto.

# Coordenadas conocidas en el plano real (por ejemplo, en metros o pixeles del mapa).
#points_labeled = name2label["001.jpg"].foot_array
points_real = np.array([
    [275, 375],
    [62.5, 273],
    [126.5, 273],
    [126.5, 213]
], dtype=np.float32)

room_outline = np.array([
    [0, 375+33.5+50],
    [0, 375+33.5],
    [195, 375+33.5],
    [195, 375],
    [0, 375],
    [0, 0],
    [293, 0],
    [293, 375],
    [293-18, 375],
    [293-18, 375+33.5],
    [293+18.5, 375+33.5],
    [293+18.5, 375+33.5+50]
], dtype=np.float32)

# Calcula la matriz de homografía.
H, _ = cv2.findHomography(points_ref, points_real)

